In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan folder output untuk laporan dan gambar sudah ada
os.makedirs('reports/gambar', exist_ok=True)
os.makedirs('reports/profil', exist_ok=True)

# 1. Memuat Data Mentah BPS (Sesuaikan path jika membaca hasil ekstraksi tahap 1 atau file raw)
# Berdasarkan manifest, file CSV BPS berada di jalur berikut:
file_path = 'C:\\Users\\yazid\\workbench-workspace\\data-wrangling\\capstone-dw-kelompok-11\\data\\raw\\bps\\2026-10-17\\sdg8_tpt.csv'

if not os.path.exists(file_path):
    # Alternatif jika membaca dari path lain atau hasil ekstraksi tahap 1
    print(f"File {file_path} tidak ditemukan, silakan sesuaikan path.")
else:
    df = pd.read_csv(file_path)
    print("Data berhasil dimuat. Kolom tersedia:", df.columns.tolist())

    # --- PENYESUAIAN KOLOM ---
    # Catatan: Sesuaikan nama kolom di bawah ini dengan struktur kolom asli di sdg8_tpt.csv miliknya BPS
    # Biasanya kolom memuat: wilayah/provinsi, periode/tahun, dan nilai (var543)
    
    # Contoh asumsi nama kolom (ubah jika berbeda di file CSV aslimu):
    # df.rename(columns={'provinsi': 'nama_wilayah', 'tahun': 'periode', 'data': 'nilai'}, inplace=True)
    
    # 2. Menghitung Statistik Deskriptif per Seri dan per Provinsi
    print("\n--- Menghitung Statistik Deskriptif ---")
    
    # Statistik keseluruhan untuk var543
    stats_global = df['nilai'].describe()
    print("Statistik Global Nilai var543:")
    print(stats_global)

    # Statistik per provinsi (jika ada kolom wilayah/provinsi)
    if 'nama_wilayah' in df.columns:
        stats_provinsi = df.groupby('nama_wilayah')['nilai'].agg(
            count='count',
            mean='mean',
            std='std',
            min='min',
            q1=lambda x: x.quantile(0.25),
            median='median',
            q3=lambda x: x.quantile(0.75),
            max='max'
        ).reset_index()
        
        # Simpan ringkasan ke CSV/Markdown di folder reports
        stats_provinsi.to_csv('reports/profil/statistik_provinsi_var543.csv', index=False)
        print("Ringkasan statistik per provinsi disimpan ke reports/profil/statistik_provinsi_var543.csv")

    # 3. Deteksi Calon Anomali (Metode IQR & Z-Score)
    print("\n--- Mendeteksi Calon Anomali ---")
    
    # Metode IQR
    Q1 = df['nilai'].quantile(0.25)
    Q3 = df['nilai'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df['anomali_iqr'] = (df['nilai'] < lower_bound) | (df['nilai'] > upper_bound)
    
    # Metode Z-Score
    mean_val = df['nilai'].mean()
    std_val = df['nilai'].std()
    df['z_score'] = (df['nilai'] - mean_val) / std_val
    df['anomali_zscore'] = abs(df['z_score']) > 3
    
    # Gabungkan calon anomali (pilih salah satu atau gabungan)
    df_anomali = df[df['anomali_iqr'] | df['anomali_zscore']].copy()
    df_anomali['alasan'] = np.where(
        df_anomali['nilai'] > upper_bound, 'Ekstrem Tinggi (> Q3 + 1.5*IQR)',
        np.where(df_anomali['nilai'] < lower_bound, 'Ekstrem Rendah (< Q1 - 1.5*IQR)', 'Z-Score > 3')
    )
    
    # Simpan daftar calon anomali
    kolom_info = [c for c in ['nama_wilayah', 'periode', 'nilai', 'alasan'] if c in df_anomali.columns]
    daftar_anomali = df_anomali[kolom_info]
    daftar_anomali.to_csv('reports/profil/daftar_calon_anomali.csv', index=False)
    print(f"Ditemukan {len(daftar_anomali)} calon anomali. Disimpan ke reports/profil/daftar_calon_anomali.csv")

    # 4. Membuat Visualisasi (Boxplot & Time Series)
    print("\n--- Membuat Grafik Visualisasi ---")
    sns.set_theme(style="whitegrid")

    # Grafik 1: Boxplot per Provinsi atau Global
    plt.figure(figsize=(12, 6))
    if 'nama_wilayah' in df.columns:
        sns.boxplot(data=df, x='nama_wilayah', y='nilai', palette='Set2')
        plt.xticks(rotation=90, fontsize=8)
    else:
        sns.boxplot(y=df['nilai'], color='skyblue')
    plt.title('Boxplot Tingkat Pengangguran Terbuka (var543)')
    plt.xlabel('Wilayah / Provinsi')
    plt.ylabel('Nilai (%)')
    plt.tight_layout()
    box_path = 'reports/gambar/boxplot_var543.png'
    plt.savefig(box_path, dpi=300)
    plt.close()
    print(f"Boxplot disimpan di {box_path}")

    # Grafik 2: Deret Waktu (Time Series)
   # Grafik 2: Deret Waktu (Time Series)
    plt.figure(figsize=(14, 6))
    if 'periode' in df.columns and 'nama_wilayah' in df.columns:
        # Plot garis per provinsi atau rata-rata nasional
        sns.lineplot(data=df, x='periode', y='nilai', hue='nama_wilayah', alpha=0.7, legend=False)
    else:
        sns.lineplot(data=df, x=df.index, y='nilai')
    plt.title('Deret Waktu Tingkat Pengangguran Terbuka (var543)')
    plt.xlabel('Periode')
    plt.ylabel('Nilai (%)')
    
    # --- PERBAIKAN DI SINI (tutup kurung ditambahkan) ---
    plt.tight_layout()
    
    ts_path = 'reports/gambar/timeseries_var543.png'
    plt.savefig(ts_path, dpi=300)
    plt.close()
    print(f"Grafik deret waktu disimpan di {ts_path}")

Data berhasil dimuat. Kolom tersedia: ['var_id', 'wilayah_id', 'wilayah', 'turvar_id', 'turvar', 'tahun_id', 'tahun', 'periode_id', 'periode', 'nilai', 'satuan']

--- Menghitung Statistik Deskriptif ---
Statistik Global Nilai var543:
count    2000.000000
mean        5.637525
std         2.925345
min         0.560000
25%         3.590000
50%         5.080000
75%         7.020000
max        18.910000
Name: nilai, dtype: float64

--- Mendeteksi Calon Anomali ---
Ditemukan 77 calon anomali. Disimpan ke reports/profil/daftar_calon_anomali.csv

--- Membuat Grafik Visualisasi ---
Boxplot disimpan di reports/gambar/boxplot_var543.png
Grafik deret waktu disimpan di reports/gambar/timeseries_var543.png
